In [3]:
import torch
import gzip
import pickle
import requests
import os
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio
from IPython.display import Markdown, display, Video
from io import BytesIO

# Delving deeper into pytorch

The goal of this assesment is to delve deeper in the fundamentals of pytorch. Due to my background we will be using regresion models of the form $f : \mathbb{R} \rightarrow \mathbb{R}$ and start working with images. 

As we will see, many of the things we have been implementing so far, are wrapped in several ways in the pytorch library, for our easy usage. Pytorch is an excellent library with many ideas copied by Tensorflow or Jax. Perhaps one of their differential ideas is the `torch.nn module` (copied by Tensorflow and implemented in Flax, which is a Jax wrapper), and their data managment tools.

Pytorch has also copied many strenghts from Jax like forward automatic differentiation and being able of obtaining gradient computational graphs as functions. While Jax has many strenghts such as jit compilation (also copied by pytorch) or XLA compilation (which I think is also now in pytorch as well).

But we should be clear that one of the fathers of the automatic differentiation engine in pytorch, which comes from the Hips lab, was hired by google to create Jax.

## Datahandling

The nice thing about pytorch is that implements different modules to allow us nice transformations over different types of data. Note that this can be used within any machine model (including sklearn) because it is just a data processing pipeline.

In its beginning we had the torchvision module, but nowdays we also have torchaudio and torchtext.

Data handling in pytorch has two main components: a dataset and a dataloader.

A dataset is in charged of all the processes involved in a data handling machine learning pipeline. These include, for example:

* Loading the data from the disk into the RAM memory. If for example the data fits in memory you might decide to load all the data. If however the data is huge you can decide having pointers to memory, and load the data just on demand. You could also want to keep the paths to the different data, or just have a chunk of your data into memory, and later on load new data from disk.
* Applying transformations to the data: normalization, feature selection, data augmentation, etc. Many of these transformation are already implemented in pytorch, with depending on the module, will be more adequate for audio, video, images or text. Obviously, pytorch allows you to include your own transformations as function pointers.

A dataloader is in charged of "asking" the dataset for new data and give this data to use it so that you can use it in your machine learning pipeline. Cool things about dataloaders include:

* Multithreading dataloading: when data is expensive to preprocess, you can configure the dataloader to use parallel computing to perform these taks in parallel.
* You can also tell the dataloader how many data you want to receive from memory (for minibatch gradient descent) and so on.
* Since the dataloader "asks" the dataset, this obviously implies that the dataloader receives, as argument, the dataset you want it to operate on.

### Datasets

Let's work with a new dataset which is call de Mnist. Mnist consists of 60 thousand 28x28 pixel grayscale images. Each of these images correspond to a number from 0 to 9. 

One of the coolest things of torchvision is that it has many famous datasets included, and mnits is one of those. You can check, for example: `https://pytorch.org/vision/stable/datasets.html`.

However, since our goal is to understand how to construct our own dataset, let's do it. As you can see in the documentation: `https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset`, a dataset is a python class where we should overwrite two methods: `__getitem__()` and `__len__()`. The earlier is the way a dataloader will tell you which sample you need to provide back, while the later is the one telling the dataloader how many samples your dataset contains. 

One cool concept introduced in torchvision is the concept of transformations. Transformations are function pointers or classes implementing a `__call__` method, that can be used by our dataset to modify the image on the fly before giving it back through the get_item method. Note that in Machine Learning we might sometimes want to perform data preprocessing just once, but it would also be nice to perform data preprocessing on the fly, for some particular tasks.

When to use a function pointer and when to use a class? Well simple, when you want the transformation to be configurable.

Let's create our custom mnist classification dataset. To do so, we need to download the data to a specific directory, and then since this data fits in memory well, our dataset can directly keep the data in RAM memory.

Download your data from ```http://www.iro.umontreal.ca/~lisa/deep/data/mnist/mnist.pkl.gz``` and load it in memory. You will load a train, validation and test split. Join the three splits into a single split for the images and the labels. So in the end you should finish with two variables X, and T keeping images and labels.

In [6]:
dataset_dir = 'mnist.pkl.gz'

try:
    with gzip.open(dataset_dir, 'rb') as f:
        try:
            train_set, valid_set, test_set = pickle.load(f, encoding='latin1')
        except:
            train_set, valid_set, test_set = pickle.load(f)
except:
    raise ValueError("Download data from http://www.iro.umontreal.ca/~lisa/deep/data/mnist/mnist.pkl.gz and place it in a directory")
    
## Read and concatenate all data into X and Y 
X_tr, Y_tr = train_set
X_va, Y_va = valid_set
X_te, Y_te = test_set

X = np.vstack((X_tr,X_te,X_va))
T = np.hstack((Y_tr,Y_te,Y_va))

Now let's create the code for our dataset. Think in a way to apply transformations in the `__getitem__` method before returning the image. 

Once done, we can create our variable keeping our dataset.

Since this mnist version of the dataset is given prepared for a fully connected neural network, let's use a reshape transformation that turns our dataset into an image that we can visualize. Since mnist is 28 by 28 pixels we need to reshape into this shape. So when instancing our class passed into transforms the function pointer to the reshape function

### Dataloader

To grab images from our dataset, we can use a dataloader, which has many usefull functions as I mentioned before. The coolest thing is that this dataloader provide us with an interable so that we can iterate over it to retrieve the full dataset. Once the full dataset has been retrieved, we can iterate again.

Let's use a batch_size of one, so that on each iteration we receive one image, and let's see the images


**Important:** In the past, we needed to apply a `transform.toTensor()` transformation in the dataset, to convert internal dataset type into torch tensors. However, it looks like this is now implicitly done by the data loader.

### Task 1: Implementing two custom transformations

Our goal will be to create different versions of the mnist dataset that we will use to train convolutional models and to learn models using self-supervision, which is the fundamental machine learning technique used by language models nowdays (such as your favourite tool chatGPT).

We will create a transformation to randomly rotate images a given angle, and a transformation to randomly mask the image using a black square.

First of all, you must think if you want this transformation configurable or not. If you want the configurable (for example the size of the mask can vary or the maximum angle of rotation) you will need to create a python instance class.

I'll make both of them configurable.

Now experiment in using these transformations. What happens if you do not reshape first?

### Task 2: Creating a pytorch dataset with our regresions problems.

Remember from our previous task we had the following regresion data.

* task $f: \mathbb{R} \rightarrow \mathbb{R}$
$$
(x_1,t_1) = (0,0.2)\\
(x_2,t_2) = (1,0.5)\\
(x_3,t_3) = (2,2.8)
$$

* task $f: \mathbb{R} \rightarrow [0,1]$
$$
\begin{split}
(x_1,t_1) &= (-0.13459237,0)\\
(x_2,t_2) &= (-3.3015387,0)\\
(x_3,t_3) &= (0.74481176,0)\\
(x_4,t_4) &= (2.62434536,1)\\
(x_5,t_5) &= (0.38824359,1)\\
(x_6,t_6) &= (0.47182825,1)\\
(x_7,t_7) &= (-0.07296862,1)\\
\end{split}
$$


* task $f: \mathbb{R}^2 \rightarrow [0,1]$
$$
\begin{split}
(x^1_{1},x^1_{2},t^1) &= (0,1,0)\\
(x^2_{1},x^2_{2},t^2) &= (1.5,2.0,0)\\
(x^3_{1},x^3_{2},t^3) &= (2,1,0)\\
(x^4_{1},x^4_{2},t^4) &= (5,3,0)\\
(x^5_{1},x^5_{2},t^5) &= (3,4,1)\\
(x^6_{1},x^6_{2},t^6) &= (4,5,1)\\
(x^7_{1},x^7_{2},t^7) &= (5,1,1)\\
\end{split}
$$

Create the three datasets containing one of these datasets and wrap them with their corresponding loaders.